# Theorem 2: Foul Up 3

> **Based on NBA play-by-play data from 2019–2024, intentionally fouling
> when leading by 3 with fewer than 12 seconds remaining shows mixed
> results — outcomes depend on time remaining and are not consistently
> better in this historical sample.**

This notebook reproduces the data collection and visualisation for Theorem 2.
Run each cell in order. Data is saved to `data/processed/` as CSV files
so that plots can be regenerated without re-running the full pipeline.

In [ ]:
# ── Imports ───────────────────────────────────────────────────────────────────────────
import sys
import pathlib

# Make sure the project root is on the path so src.* imports work.
ROOT = pathlib.Path().resolve().parent  # notebooks/ -> project root
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import matplotlib
matplotlib.use("Agg")  # headless; swap to "inline" in a live Jupyter kernel
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns

PROCESSED_DIR = ROOT / "data" / "processed"
IMAGES_DIR    = ROOT / "docs" / "assets" / "images"

print("Project root  :", ROOT)
print("Processed dir :", PROCESSED_DIR)

## Step 1 – Collect data

Run `collect()` to scan the historical play-by-play log for situations
where the home team leads by 3 with fewer than 12 seconds remaining.
Win rates are computed for **foul** vs. **no-foul** defence and saved
as CSV grids in `data/processed/`.

Files written:

| File | Contents |
|------|----------|
| `theorem2_grid.csv` | Win-rate gain grid (foul − no-foul) |
| `theorem2_wp_foul_grid.csv` | Historical win rate when fouling |
| `theorem2_wp_no_foul_grid.csv` | Historical win rate without fouling |
| `theorem2_metadata.json` | Axis labels (time values, FG3% values) |

> *Skip this cell if those files already exist in `data/processed/`.*

In [ ]:
from src.theorems.theorem2 import collect

grid_path, meta_path = collect(out_dir=PROCESSED_DIR, processed_dir=PROCESSED_DIR)
print("Saved grid :", grid_path)
print("Saved meta :", meta_path)

## Step 2 – Load grid data

`load_grids()` reads the CSV files and returns the gain grid, the two
underlying win-rate grids, and the axis labels.

In [ ]:
from src.theorems.theorem2 import load_grids

gain_grid, wp_foul_grid, wp_no_foul_grid, time_values, fg3_values = load_grids(PROCESSED_DIR)

print("Grid shape  :", gain_grid.shape)
print("Time values :", time_values)
print("FG3 values  :", [f'{v:.0%}' for v in fg3_values])
print()
print("Gain grid (win % gain from fouling, rows=time, cols=FG3%):")
print(np.round(gain_grid * 100, 2))

## Step 3 – Reproduce the heatmap

`plot()` generates a heatmap of the win-percentage gain from fouling
across time remaining (y-axis) and opponent 3-point field-goal percentage
(x-axis). Green cells indicate that fouling is historically better;
red cells indicate that normal defence is better.

In [ ]:
from src.theorems.theorem2 import plot

IMAGES_DIR.mkdir(parents=True, exist_ok=True)

fig_path = plot(processed_dir=PROCESSED_DIR, images_dir=IMAGES_DIR)
print("Saved figure:", fig_path)

## Step 4 – Display the heatmap inline

In [ ]:
from src.theorems.theorem2 import load_grids, PALETTE, FONT_FAMILY

gain_grid, _, _, time_values, fg3_values = load_grids(PROCESSED_DIR)

display_grid = gain_grid * 100
x_labels = [f'{v:.0%}' for v in fg3_values]
y_labels = [f'{t}s' for t in time_values]

plt.rcParams.update({
    "font.family": FONT_FAMILY,
    "axes.titlesize": 14,
    "axes.labelsize": 12,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,
})

fig, ax = plt.subplots(figsize=(10, 6))
sns.heatmap(
    display_grid,
    ax=ax,
    xticklabels=x_labels,
    yticklabels=y_labels,
    cmap=PALETTE,
    center=0.0,
    annot=True,
    fmt=".1f",
    linewidths=0.4,
    linecolor="white",
    cbar_kws={"label": "Historical Win % Gain from Fouling (pp)", "shrink": 0.85},
)
ax.set_title(
    "Theorem 2: Foul Up 3\n"
    "Historical Win % Gain from Intentional Foul vs. Normal Defense",
    fontweight="bold",
    pad=16,
)
ax.set_xlabel("Opponent 3-Point Field Goal %", labelpad=10)
ax.set_ylabel("Seconds Remaining", labelpad=10)
ax.text(
    0.5, -0.14,
    "Green = Fouling is better  |  Red = Normal defense is better  |  Values in percentage points",
    ha="center", va="center", transform=ax.transAxes, fontsize=9, color="gray",
)
plt.tight_layout()
plt.show()

## Step 5 – Generate the Markdown documentation

In [ ]:
from src.theorems.theorem2 import generate_doc

DOCS_DIR = ROOT / "docs"
doc_path = generate_doc(processed_dir=PROCESSED_DIR, docs_dir=DOCS_DIR)
print("Saved doc:", doc_path)